In [2]:
"""Interactive maps of ANFR radio supports for 6 French overseas territories."""
import pandas as pd
import folium

# Satellite basemap (Esri). Default OpenStreetMap tiles are not used:
# they are blocked when the HTML file is opened by double-click.
BASEMAP = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"

# Start of the commune INSEE code -> territory name (used for the file names)
TERRITORIES = {
    "974": "reunion",
    "971": "guadeloupe",
    "972": "martinique",
    "973": "guyane",
    "976": "mayotte",
    "988": "nouvelle_caledonie",
}


def read_table(folder, name, columns):
    """Read one ANFR text file, keeping only the columns we need.
    STA_NM_ANFR and COM_CD_INSEE are read as text: some values (e.g. 0422750464)
    are not plain numbers, and all tables must use the same type to be joined."""
    return pd.read_csv(f"{folder}/{name}.txt", sep=";", encoding="utf-8",
                       usecols=columns, dtype={"STA_NM_ANFR": str, "COM_CD_INSEE": str})


def to_decimal(degrees, minutes, seconds, direction):
    """Degrees / minutes / seconds -> decimal degrees (South and West are negative)."""
    value = degrees + minutes / 60 + seconds / 3600
    return value.where(~direction.isin(["S", "W"]), -value)


# 1. Supports (one row per support and installation) with decimal coordinates
support = read_table("data", "sup_support",
                     ["SUP_ID", "STA_NM_ANFR", "NAT_ID", "SUP_NM_HAUT", "COM_CD_INSEE",
                      "COR_NB_DG_LAT", "COR_NB_MN_LAT", "COR_NB_SC_LAT", "COR_CD_NS_LAT",
                      "COR_NB_DG_LON", "COR_NB_MN_LON", "COR_NB_SC_LON", "COR_CD_EW_LON"])
support["lat"] = to_decimal(support["COR_NB_DG_LAT"], support["COR_NB_MN_LAT"],
                            support["COR_NB_SC_LAT"], support["COR_CD_NS_LAT"])
support["lon"] = to_decimal(support["COR_NB_DG_LON"], support["COR_NB_MN_LON"],
                            support["COR_NB_SC_LON"], support["COR_CD_EW_LON"])

# 2. Keep only our 6 territories (first 3 characters of the INSEE code)
support["code"] = support["COM_CD_INSEE"].str[:3]
support = support[support["code"].isin(list(TERRITORIES))].dropna(subset=["lat", "lon"])

# 3. Add the operator, the service date and the nature of the support
station = read_table("data", "sup_station", ["STA_NM_ANFR", "ADM_ID", "DTE_EN_SERVICE"])
exploitant = read_table("ref", "sup_exploitant", ["ADM_ID", "ADM_LB_NOM"])
nature = read_table("ref", "sup_nature", ["NAT_ID", "NAT_LB_NOM"])
installations = (support
                 .merge(station, on="STA_NM_ANFR", how="left")
                 .merge(exploitant, on="ADM_ID", how="left")
                 .merge(nature, on="NAT_ID", how="left"))

# 4. Add the antenna types, as one text per installation (e.g. "Panneau, Tube")
antenne = read_table("data", "sup_antenne", ["STA_NM_ANFR", "SUP_ID", "TAE_ID"])
antenne = antenne[antenne["STA_NM_ANFR"].isin(support["STA_NM_ANFR"])]   # our territories only
antenne = antenne.merge(read_table("ref", "sup_type_antenne", ["TAE_ID", "TAE_LB"]), on="TAE_ID", how="left")
antenne = antenne.dropna(subset=["TAE_LB"]).drop_duplicates(["STA_NM_ANFR", "SUP_ID", "TAE_LB"]).sort_values("TAE_LB")
antennes = antenne.groupby(["STA_NM_ANFR", "SUP_ID"])["TAE_LB"].agg(", ".join).reset_index(name="antennes")
installations = installations.merge(antennes, on=["STA_NM_ANFR", "SUP_ID"], how="left")

installations = installations.fillna("inconnu")   # missing values are shown as "inconnu"
print("Installations:", len(installations), "| Supports:", installations["SUP_ID"].nunique())
print("Operateur inconnu:", (installations["ADM_LB_NOM"] == "inconnu").sum())   # check: joins worked

# 5. One map per territory, one marker per support
for code, name in TERRITORIES.items():
    data = installations[installations["code"] == code]
    print(name, ":", data["SUP_ID"].nunique(), "supports")
    if data.empty:
        continue                                   # territory missing from the data

    carte = folium.Map(tiles=BASEMAP, attr="Tiles © Esri")
    for sup_id, group in data.groupby("SUP_ID"):
        first = group.iloc[0]
        text = f"<b>Support {sup_id}</b><br>Nature : {first['NAT_LB_NOM']}<br>Hauteur (m) : {first['SUP_NM_HAUT']}"
        for _, row in group.iterrows():            # one block per installation on this support
            text += (f"<hr>Opérateur : {row['ADM_LB_NOM']}"
                     f"<br>Antennes : {row['antennes']}"
                     f"<br>Mise en service : {row['DTE_EN_SERVICE']}")
        folium.CircleMarker([first["lat"], first["lon"]], radius=3,
                            popup=folium.Popup(text, max_width=300)).add_to(carte)

    carte.fit_bounds([[data["lat"].min(), data["lon"].min()],
                      [data["lat"].max(), data["lon"].max()]])
    carte.save(f"carte_{name}.html")

Installations: 7074 | Supports: 4411
Operateur inconnu: 0
reunion : 1496 supports
guadeloupe : 602 supports
martinique : 601 supports
guyane : 490 supports
mayotte : 275 supports
nouvelle_caledonie : 947 supports
